In [1]:
# Importing ChexFound Code Base

import os
import sys
import importlib

# 1. Setup repository path
repo_dir = '/kaggle/working/CheXFound'
if not os.path.exists(os.path.join(repo_dir, 'chexfound', '__init__.py')):
    print("Cloning CheXFound codebase...")
    !rm -rf /kaggle/working/CheXFound
    !git clone https://github.com/SuyogMagar/CheXFound.git /kaggle/working/CheXFound

# 2. Register path permanently for this notebook session
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

importlib.invalidate_caches()

# 3. Test import to guarantee future cells will not fail
from chexfound.models import vision_transformer as vits
print("\n✓ SUCCESS: CheXFound codebase is imported, verified, and locked in sys.path.")

Cloning CheXFound codebase...
Cloning into '/kaggle/working/CheXFound'...
remote: Enumerating objects: 460, done.
remote: Counting objects: 100% (460/460), done.
remote: Compressing objects: 100% (358/358), done.
remote: Total 460 (delta 134), reused 407 (delta 96), pack-reused 0 (from 0)
Receiving objects: 100% (460/460), 24.14 MiB | 32.61 MiB/s, done.
Resolving deltas: 100% (134/134), done.

✓ SUCCESS: CheXFound codebase is imported, verified, and locked in sys.path.


/kaggle/working/CheXFound/chexfound/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/kaggle/working/CheXFound/chexfound/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/kaggle/working/CheXFound/chexfound/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


In [2]:
# Dataset Verification, Head Inspection

import os
import glob
import pandas as pd

# 1. Locate dataset CSV
possible_csv = glob.glob('/kaggle/input/**/Data_Entry_2017*.csv', recursive=True)
assert len(possible_csv) > 0, "Could not find Data_Entry_2017.csv in /kaggle/input/"
csv_path = possible_csv[0]
print(f"Found Metadata CSV: {csv_path}")

df = pd.read_csv(csv_path)

# 2. Map image index to full filesystem path across all image folders
all_images = {os.path.basename(p): p for p in glob.glob('/kaggle/input/**/*.png', recursive=True)}
df['full_path'] = df['Image Index'].map(all_images)
df = df.dropna(subset=['full_path']).reset_index(drop=True)

# 3. 14 standard thorax pathologies in NIH-14
CLASSES = [
    'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Nodule',
    'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema', 'Emphysema',
    'Fibrosis', 'Pleural_Thickening', 'Hernia'
]

for c in CLASSES:
    df[c] = df['Finding Labels'].apply(lambda x: 1 if c in str(x).split('|') else 0)

# 4. Official train/test split using test_list.txt
test_list_files = glob.glob('/kaggle/input/**/test_list.txt', recursive=True)
assert len(test_list_files) > 0, "Could not find test_list.txt"
with open(test_list_files[0]) as f:
    test_set_names = set(line.strip() for line in f)

train_df = df[~df['Image Index'].isin(test_set_names)].reset_index(drop=True)
test_df = df[df['Image Index'].isin(test_set_names)].reset_index(drop=True)

print(f"\n✓ Verified Images Available: {len(df)}")
print(f"✓ Train Pool: {len(train_df)} scans | Test Pool: {len(test_df)} scans")

# Display the head of the processed dataset
df[['Image Index', 'Finding Labels'] + CLASSES[:5]].head(5)

Found Metadata CSV: /kaggle/input/datasets/nih-chest-xrays/data/Data_Entry_2017.csv

✓ Verified Images Available: 112120
✓ Train Pool: 86524 scans | Test Pool: 25596 scans


,Image Index,Finding Labels,Atelectasis,Cardiomegaly,Effusion,Infiltration,Mass
0,00000001_000.png,Cardiomegaly,0,1,0,0,0
1,00000001_001.png,Cardiomegaly|Emphysema,0,1,0,0,0
2,00000001_002.png,Cardiomegaly|Effusion,0,1,1,0,0
3,00000002_000.png,No Finding,0,0,0,0,0
4,00000003_000.png,Hernia,0,0,0,0,0


In [3]:
# Load and Freeze the ChexFound Trained Parameters

import os
import glob
import torch
import torch.nn as nn
from chexfound.models import vision_transformer as vits

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device for inference: {device}")

# 1. Locate teacher checkpoint
teacher_files = glob.glob('/kaggle/input/**/teacher_checkpoint.pth', recursive=True)
assert len(teacher_files) > 0, "Could not find teacher_checkpoint.pth in /kaggle/input/"
teacher_path = teacher_files[0]
print(f"Loading weights from: {teacher_path}")

# 2. Instantiate ViT-Large backbone (512x512 resolution, patch_size=16)
model = vits.vit_large(patch_size=16, img_size=512)

# 3. Load checkpoint and align state dict keys
teacher_ckpt = torch.load(teacher_path, map_location='cpu')
state_dict = teacher_ckpt.get('teacher', teacher_ckpt)
clean_state_dict = {
    (k[9:] if k.startswith('backbone.') else k): v 
    for k, v in state_dict.items()
}

missing_keys, unexpected_keys = model.load_state_dict(clean_state_dict, strict=False)

# 4. Freeze backbone and mount on GPU
model.eval()
model.to(device)
for param in model.parameters():
    param.requires_grad = False

print(f"\n✓ CheXFound ViT-Large Loaded:")
print(f"  - Missing keys: {len(missing_keys)}")
print(f"  - Unexpected keys: {len(unexpected_keys)}")
print(f"  - All parameters frozen: {all(not p.requires_grad for p in model.parameters())}")

# 5. Sanity check: Run a dummy forward pass through the last 4 blocks
dummy = torch.randn(1, 3, 512, 512).to(device)
with torch.no_grad():
    sample_layers = model.get_intermediate_layers(dummy, n=4, return_class_token=True)
    # Each block outputs: (patch_tokens [B, 1024, 1024], cls_token [B, 1024])
    patch_sample, cls_sample = sample_layers[-1]
    
print(f"\n✓ Forward check successful:")
print(f"  - Patch tokens shape: {patch_sample.shape} (1024 spatial patches)")
print(f"  - CLS token shape   : {cls_sample.shape} (1024 dimensions)")

Device for inference: cuda
Loading weights from: /kaggle/input/datasets/suyogmagar08/chexfound-weights/teacher_checkpoint.pth

✓ CheXFound ViT-Large Loaded:
  - Missing keys: 240
  - Unexpected keys: 305
  - All parameters frozen: True

✓ Forward check successful:
  - Patch tokens shape: torch.Size([1, 1024, 1024]) (1024 spatial patches)
  - CLS token shape   : torch.Size([1, 1024]) (1024 dimensions)


In [4]:
import os
import sys
import glob
import importlib
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from PIL import Image
import torchvision.transforms as T
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# 1. Setup repository path & import ViT architecture
repo_dir = '/kaggle/working/CheXFound'
if not os.path.exists(os.path.join(repo_dir, 'chexfound', '__init__.py')):
    !rm -rf /kaggle/working/CheXFound
    !git clone https://github.com/SuyogMagar/CheXFound.git /kaggle/working/CheXFound

if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

importlib.invalidate_caches()
from chexfound.models import vision_transformer as vits

# 2. Load Frozen CheXFound ViT-Large Backbone
teacher_files = glob.glob('/kaggle/input/**/teacher_checkpoint.pth', recursive=True)
assert len(teacher_files) > 0, "teacher_checkpoint.pth not found in /kaggle/input/"

model = vits.vit_large(patch_size=16, img_size=512)
teacher_ckpt = torch.load(teacher_files[0], map_location='cpu')
state_dict = teacher_ckpt.get('teacher', teacher_ckpt)
clean_state_dict = {(k[9:] if k.startswith('backbone.') else k): v for k, v in state_dict.items()}
model.load_state_dict(clean_state_dict, strict=False)
model.eval()
model.to(device)
for p in model.parameters():
    p.requires_grad = False
print("✓ CheXFound ViT-Large backbone loaded and frozen.")

# 3. Locate all Shenzhen CXR Scans
search_paths = [
    '/kaggle/input/**/CXR_png/*.png',
    '/kaggle/input/**/images/*.png',
    '/kaggle/input/**/*.png'
]

tb_images = []
for sp in search_paths:
    found = glob.glob(sp, recursive=True)
    tb_images = [p for p in found if 'CHN' in os.path.basename(p) or 'CHX' in os.path.basename(p)]
    if len(tb_images) > 0:
        break

assert len(tb_images) > 0, "Shenzhen images not found. Ensure dataset is added via + Add Input."

records = []
for p in tb_images:
    base = os.path.splitext(os.path.basename(p))[0]
    label = 1 if base.endswith('_1') else 0
    records.append({'path': p, 'label': label})

df_all = pd.DataFrame(records).sample(frac=1.0, random_state=42).reset_index(drop=True)
print(f"Total Scans Indexed: {len(df_all)} (Normal: {(df_all['label']==0).sum()}, TB: {(df_all['label']==1).sum()})")

# 4. Extract Per-Layer L2-Normalized CLS Tokens (Layers 21, 22, 23, 24)
eval_transform = T.Compose([
    T.Resize((512, 512)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class ShenzhenDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        img = Image.open(self.df.iloc[idx]['path']).convert('RGB')
        return self.transform(img), self.df.iloc[idx]['label']

loader = DataLoader(ShenzhenDataset(df_all, eval_transform), batch_size=16, shuffle=False, num_workers=2)

print("\n--- Extracting Multi-Block CLS Representations ---")
all_cls_normed = []
all_labels = []

with torch.no_grad():
    for imgs, lbls in tqdm(loader, desc="Extraction"):
        imgs = imgs.to(device)
        layers = model.get_intermediate_layers(imgs, n=4, return_class_token=True)
        # Normalize each block's CLS token independently to equalize feature magnitude
        normed_cls = [F.normalize(block[1], p=2, dim=-1) for block in layers]
        cls_concat = torch.cat(normed_cls, dim=-1)  # [B, 4096]
        all_cls_normed.append(cls_concat.cpu())
        all_labels.append(lbls)

X_all = torch.cat(all_cls_normed, dim=0)
y_all = torch.cat(all_labels, dim=0)
print(f"Extracted Feature Matrix: {X_all.shape}")

# 5. 5-Fold Stratified Cross-Validation for Linear Probe
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_aurocs, fold_auprcs = [], []

print("\n" + "="*58)
print("     TABLE IV: LINEAR PROBE BENCHMARK (SHENZHEN)")
print("="*58)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_all, y_all)):
    X_f_tr, y_f_tr = X_all[train_idx], y_all[train_idx]
    X_f_val, y_f_val = X_all[val_idx], y_all[val_idx]
    
    fold_train_loader = DataLoader(
        TensorDataset(X_f_tr, y_f_tr.float().unsqueeze(1)),
        batch_size=32,
        shuffle=True
    )
    
    probe = nn.Sequential(
        nn.BatchNorm1d(4096),
        nn.Linear(4096, 1)
    ).to(device)
    
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(probe.parameters(), lr=1e-3, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=80, eta_min=1e-6)
    
    # 80 epochs per fold (~1 second each on cached tensors)
    for epoch in range(80):
        probe.train()
        for bx, by in fold_train_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(probe(bx), by)
            loss.backward()
            optimizer.step()
        scheduler.step()
        
    probe.eval()
    with torch.no_grad():
        val_logits = probe(X_f_val.to(device))
        val_probs = torch.sigmoid(val_logits).cpu().squeeze().numpy()
        
    y_true = y_f_val.numpy()
    roc = roc_auc_score(y_true, val_probs)
    prc = average_precision_score(y_true, val_probs)
    
    fold_aurocs.append(roc)
    fold_auprcs.append(prc)
    print(f"Fold {fold+1}/5 | AUROC: {roc:.4f} | AUPRC: {prc:.4f}")

mean_roc = np.mean(fold_aurocs)
std_roc = np.std(fold_aurocs)
mean_prc = np.mean(fold_auprcs)

print("-"*58)
print(f"Linear Probe AUROC : {mean_roc:.4f} ± {std_roc:.4f}  (Table IV: 0.967)")
print(f"Linear Probe AUPRC : {mean_prc:.4f}              (Table IV: 0.974)")
print("="*58)

Device: cuda
✓ CheXFound ViT-Large backbone loaded and frozen.
Total Scans Indexed: 662 (Normal: 326, TB: 336)

--- Extracting Multi-Block CLS Representations ---


Extraction: 100%|██████████| 42/42 [03:15<00:00,  4.67s/it]


Extracted Feature Matrix: torch.Size([662, 4096])

     TABLE IV: LINEAR PROBE BENCHMARK (SHENZHEN)
Fold 1/5 | AUROC: 0.8260 | AUPRC: 0.8693
Fold 2/5 | AUROC: 0.9030 | AUPRC: 0.9195
Fold 3/5 | AUROC: 0.8602 | AUPRC: 0.8671
Fold 4/5 | AUROC: 0.8879 | AUPRC: 0.8926
Fold 5/5 | AUROC: 0.8425 | AUPRC: 0.8618
----------------------------------------------------------
Linear Probe AUROC : 0.8639 ± 0.0283  (Table IV: 0.967)
Linear Probe AUPRC : 0.8820              (Table IV: 0.974)
